# SMART FHIR API Ingestion

## Purpose

This notebook ingests synthetic healthcare data from the SMART Health IT
FHIR R4 API into the Bronze layer of the Khaoula Healthy Insurance Data Platform.

### API Resources

- Patient
- Condition
- Encounter

### Targets

- `health_insurance.bronze.fhir_patient_raw`
- `health_insurance.bronze.fhir_condition_raw`
- `health_insurance.bronze.fhir_encounter_raw`



In [0]:

# Project and API configuration


CATALOG = "health_insurance"
BRONZE_SCHEMA = "bronze"

FHIR_BASE_URL = "https://r4.smarthealthit.org"

FHIR_RESOURCES = [
    "Patient",
    "Condition",
    "Encounter"
]

PAGE_SIZE = 100

print("FHIR API:", FHIR_BASE_URL)
print("Resources:", FHIR_RESOURCES)
print("Page size:", PAGE_SIZE)

In [0]:
import requests

test_url = f'{FHIR_BASE_URL}/Patient?_count=3'

response = requests.get(
    test_url,
    headers={'accept':'application/fhir+json'},
    timeout=30
)

print("HTTP status:", response.status_code)

response.raise_for_status()

test_payload = response.json()

print("FHIR resource type:", test_payload.get("resourceType"))
print("Bundle type:", test_payload.get("type"))
print("Entries returned:", len(test_payload.get("entry", [])))


In [0]:

# Inspect FHIR Bundle structure


print("Top-level Bundle fields:")
print(list(test_payload.keys()))

print("\nBundle links:")

for link in test_payload.get("link", []):
    print(
        link.get("relation"),
        "->",
        link.get("url")
    )

In [0]:

# Helper: extract the next-page URL from a FHIR Bundle


def get_next_link(bundle):
    """
    Return the URL of the next page from a FHIR Bundle.

    Returns None when no additional page exists.
    """
    
    for link in bundle.get("link", []):
        if link.get("relation") == "next":
            return link.get("url")
    
    return None

In [0]:
# ttesting the function

next_url = get_next_link(test_payload)

print("Next page:")
print(next_url)

In [0]:
# ============================================================
# Reusable FHIR resource extractor
# ============================================================

def fetch_fhir_resource(
    resource_type,
    page_size=100,
    max_pages=None
):
    """
    Retrieve a FHIR resource from SMART Health IT.

    Handles FHIR Bundle pagination and returns the raw
    resource objects contained in Bundle entries.

    Parameters
    ----------
    resource_type : str
        Example: Patient, Condition, Encounter

    page_size : int
        Number of records requested per API page.

    max_pages : int or None
        Optional development limit.
        None means continue until pagination ends.
    """

    url = (
        f"{FHIR_BASE_URL}/{resource_type}"
        f"?_count={page_size}"
    )

    records = []
    page_number = 0

    while url:

        page_number += 1

        print(
            f"Fetching {resource_type} "
            f"page {page_number}..."
        )

        response = requests.get(
            url,
            headers={"Accept": "application/fhir+json"},
            timeout=60
        )

        response.raise_for_status()

        bundle = response.json()

        entries = bundle.get("entry", [])

        for entry in entries:
            resource = entry.get("resource")

            if resource:
                records.append(resource)

        print(
            f"  Records collected: {len(records)}"
        )

        if (
            max_pages is not None
            and page_number >= max_pages
        ):
            break

        url = get_next_link(bundle)

    return records

In [0]:

# Development test — Patient


patient_sample = fetch_fhir_resource(
    resource_type="Patient",
    page_size=100,
    max_pages=2
)

print(
    "Patient records retrieved:",
    len(patient_sample)
)

In [0]:
# inspecting a record

patient_sample[0]

In [0]:

# Convert FHIR resources to Bronze DataFrame


import json
from pyspark.sql import functions as F


def create_fhir_bronze_df(records, resource_type):
    """
    Convert raw FHIR resource dictionaries into a Bronze
    Spark DataFrame while preserving the original JSON.
    """

    json_rows = [
        (json.dumps(record),)
        for record in records
    ]

    df = spark.createDataFrame(
        json_rows,
        ["raw_json"]
    )

    return (
        df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_system", F.lit("smart_fhir"))
        .withColumn("_resource_type", F.lit(resource_type))
    )

In [0]:

# configuring the FHIR ingestion targets


FHIR_TARGETS = {
    "Patient": "fhir_patient_raw",
    "Condition": "fhir_condition_raw",
    "Encounter": "fhir_encounter_raw"
}

for resource, table in FHIR_TARGETS.items():
    print(f"{resource} -> {CATALOG}.{BRONZE_SCHEMA}.{table}")

In [0]:
# defining data ingestion limits

PAGE_SIZE = 100
MAX_PAGES = 5


# Extract FHIR resources


fhir_records = {}

for resource_type in FHIR_TARGETS:

    print(f"\n{'=' * 60}")
    print(f"INGESTING: {resource_type}")
    print(f"{'=' * 60}")

    records = fetch_fhir_resource(
        resource_type=resource_type,
        page_size=PAGE_SIZE,
        max_pages=MAX_PAGES
    )

    fhir_records[resource_type] = records

    print(
        f"Completed {resource_type}: "
        f"{len(records):,} records"
    )

In [0]:

# Writing FHIR resources to Bronze Delta tables


for resource_type, table_name in FHIR_TARGETS.items():

    records = fhir_records[resource_type]

    bronze_df = create_fhir_bronze_df(
        records=records,
        resource_type=resource_type
    )

    target_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(
        f"{resource_type}: "
        f"{len(records):,} records -> {target_table}"
    )

In [0]:
%sql
--testing the the newly written tables

SELECT 'Patient' AS resource, COUNT(*) AS records
FROM health_insurance.bronze.fhir_patient_raw

UNION ALL

SELECT 'Condition', COUNT(*)
FROM health_insurance.bronze.fhir_condition_raw

UNION ALL

SELECT 'Encounter', COUNT(*)
FROM health_insurance.bronze.fhir_encounter_raw;

## Ingestion Result

FHIR resources were successfully retrieved from the SMART Health IT
FHIR R4 API using paginated HTTP requests and persisted to Unity
Catalog as Bronze Delta tables.

### Bronze tables

- `health_insurance.bronze.fhir_patient_raw`
- `health_insurance.bronze.fhir_condition_raw`
- `health_insurance.bronze.fhir_encounter_raw`

### Bronze design

FHIR resources are stored as raw JSON to preserve their original
nested structure.

Only technical ingestion metadata is added:

- `_ingested_at`
- `_source_system`
- `_resource_type`

FHIR structures will be parsed, flattened, typed, and standardized
during Silver-layer transformation.